In [1]:
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("gold").master("local[*]").getOrCreate()

SILVER = "/home/work/data/silver/"
GOLD = "/home/work/data/gold/"

def lire_silver(nom):
    return spark.read.parquet(SILVER + nom)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 19:09:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
oi = lire_silver("order_items")
pr = lire_silver("products")
se = lire_silver("sellers")
cu = lire_silver("customers")
oe = lire_silver("orders_enriched")

print("order_items : lignes                 :", oi.count(), "(attendu 112626)")
print("order_items : couples distincts      :",
      oi.select("order_id", "order_item_id").distinct().count(), "(doit égaler la ligne précédente)")
print("products : lignes / product_id distincts :",
      pr.count(), "/", pr.select("product_id").distinct().count(), "(attendu 32951 / 32951)")
print("sellers  : lignes / seller_id distincts  :",
      se.count(), "/", se.select("seller_id").distinct().count(), "(attendu 3095 / 3095)")
print("customers : lignes / customer_id / customer_unique_id :",
      cu.count(), "/", cu.select("customer_id").distinct().count(), "/",
      cu.select("customer_unique_id").distinct().count(), "(attendu 99441 / 99441 / 96096)")

print("\nOrphelins (attendu 0 partout) :")
print("  order_items -> products :", oi.join(pr, "product_id", "left_anti").count())
print("  order_items -> sellers  :", oi.join(se, "seller_id", "left_anti").count())
print("  order_items -> orders   :", oi.join(oe.select("order_id"), "order_id", "left_anti").count())

order_items : lignes                 : 112626 (attendu 112626)


order_items : couples distincts      : 112626 (doit égaler la ligne précédente)
products : lignes / product_id distincts : 32951 / 32951 (attendu 32951 / 32951)
sellers  : lignes / seller_id distincts  : 3095 / 3095 (attendu 3095 / 3095)
customers : lignes / customer_id / customer_unique_id : 99441 / 99441 / 96096 (attendu 99441 / 99441 / 96096)

Orphelins (attendu 0 partout) :
  order_items -> products : 0
  order_items -> sellers  : 0
  order_items -> orders   : 0


In [3]:
par_personne = (cu.groupBy("customer_unique_id")
    .agg(F.count("*").alias("n_commandes"),
         F.countDistinct("customer_state").alias("n_etats"),
         F.countDistinct("customer_city").alias("n_villes")))

print("Personnes avec plusieurs commandes :", par_personne.filter("n_commandes > 1").count())
print("Personnes avec plusieurs États     :", par_personne.filter("n_etats > 1").count())
print("Personnes avec plusieurs villes    :", par_personne.filter("n_villes > 1").count())

par_zip = (cu.groupBy("customer_zip_code_prefix")
    .agg(F.countDistinct("customer_state").alias("n_etats"),
         F.countDistinct("customer_city").alias("n_villes")))

print("\nCodes postaux clients                 :", par_zip.count())
print("Codes postaux avec plusieurs États    :", par_zip.filter("n_etats > 1").count())
print("Codes postaux avec plusieurs villes   :", par_zip.filter("n_villes > 1").count())

Personnes avec plusieurs commandes : 2997
Personnes avec plusieurs États     : 39
Personnes avec plusieurs villes    : 122

Codes postaux clients                 : 14994
Codes postaux avec plusieurs États    : 0
Codes postaux avec plusieurs villes   : 39


In [4]:
o = lire_silver("orders")
o.agg(F.min("order_purchase_timestamp").alias("achat_min"),
      F.max("order_purchase_timestamp").alias("achat_max"),
      F.min("order_estimated_delivery_date").alias("estim_min"),
      F.max("order_estimated_delivery_date").alias("estim_max"),
      F.max("order_delivered_customer_date").alias("livr_max")).show(truncate=False)

base = oi.join(oe.select("order_id", "is_sale", "in_analysis_window", "order_status"),
               "order_id", "left")

print("Lignes après jointure          :", base.count(), "(doit rester 112626)")
print("Lignes sans commande associée  :", base.filter("is_sale IS NULL").count(), "(attendu 0)")
print(f"CA Silver  : {base.agg(F.sum('price')).first()[0]:,.2f} (attendu 13 588 545,09)")
print(f"CA ventes  : {base.filter('is_sale').agg(F.sum('price')).first()[0]:,.2f} (attendu 13 491 302,13)")
print("Lignes produit dans la fenêtre d'analyse :", base.filter("in_analysis_window").count())

+-------------------+-------------------+-------------------+-------------------+-------------------+
|achat_min          |achat_max          |estim_min          |estim_max          |livr_max           |
+-------------------+-------------------+-------------------+-------------------+-------------------+
|2016-09-04 21:15:19|2018-10-17 17:30:18|2016-09-30 00:00:00|2018-11-12 00:00:00|2018-10-17 13:22:46|
+-------------------+-------------------+-------------------+-------------------+-------------------+

Lignes après jointure          : 112626 (doit rester 112626)
Lignes sans commande associée  : 0 (attendu 0)
CA Silver  : 13,588,545.09 (attendu 13 588 545,09)
CA ventes  : 13,491,302.13 (attendu 13 491 302,13)
Lignes produit dans la fenêtre d'analyse : 112255
